# OrganBagTransformer (OBT) — İnteraktif Viewer

CT kesiti + Organ bag sınırları + FCOS tahminleri + Hasta-düzeyi çıktılar.

| Panel | İçerik |
|---|---|
| **Sol** | CT ham + GT kutuları + FCOS tahmin kutuları |
| **Orta** | Dilim önem haritası (organ bag sınırları + mevcut z işareti) |
| **Sağ** | Hasta olasılıkları + triyaj + belirsizlik + cross-organ attention |

> **Gereksinim:** `Faz3c_OrganBagTransformer_Colab_Kaggle.ipynb` çıktıları (`obt_fold{N}/output/`) mevcut olmalı.

In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'], check=True)

CompletedProcess(args=['C:\\Users\\ramazan.polat3\\AppData\\Local\\Microsoft\\WindowsApps\\PythonSoftwareFoundation.Python.3.8_qbz5n2kfra8p0\\python.exe', '-m', 'pip', 'install', '-q', 'ipywidgets'], returncode=0)

In [5]:
import os, sys, json, warnings
import threading
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from PIL import Image
from ipywidgets import (
    IntSlider, Dropdown, HBox, VBox, Output, ToggleButton, Label, HTML
)
from IPython.display import display

warnings.filterwarnings('ignore')

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DATA_ROOT = Path(os.environ.get('TR_ABDOMEN_BASE',  r'D:/makale-pdf/Proje/abdomen'))
PROJE     = Path(os.environ.get('TR_ABDOMEN_PROJE', r'D:/makale-pdf/Proje'))
CODE      = Path(os.environ.get('TR_ABDOMEN_CODE',  r'D:/makale-pdf/Proje/code'))

sys.path.insert(0, str(CODE))
from src.config import DET_DATA_DIR, SUPER_CLASSES
from src.splits import load_fold

FOLD = 0
FOLD_DIR  = DET_DATA_DIR / f'fold{FOLD}'
OUT_DIR   = PROJE / 'outputs' / f'obt_fold{FOLD}' / 'output'

ANATOMICAL_CLASSES = [
    'Abdominal Aorta', 'Gall bladder', 'Pancreas',
    'Kidney-Bladder', 'Colon', 'appendix',
]
# Patoloji sınıfı → anatomik organ indeksi
CLASS_TO_ORGAN_IDX = [1, 3, 2, 0, 5, 4]

CLASS_COLORS = [
    '#e74c3c', '#3498db', '#2ecc71',
    '#f39c12', '#9b59b6', '#1abc9c',
]
ORGAN_COLORS = [
    '#e74c3c', '#f39c12', '#2ecc71',
    '#3498db', '#9b59b6', '#1abc9c',
]
TRIAGE_COLORS = {'normal': '#27ae60', 'gözlem': '#f39c12', 'acil': '#e74c3c'}

# ── Çıktı dosyaları ─────────────────────────────────────────────────────────
_fcos_csv    = OUT_DIR / f'obt_fold{FOLD}_fcos_pred.csv'
_patient_csv = OUT_DIR / f'obt_fold{FOLD}_patient_pred.csv'
_attn_json   = OUT_DIR / f'obt_fold{FOLD}_organ_attn.json'
_xorgan_json = OUT_DIR / f'obt_fold{FOLD}_cross_organ_attn.json'

FCOS_DF    = pd.read_csv(_fcos_csv)    if _fcos_csv.exists()    else pd.DataFrame()
PATIENT_DF = pd.read_csv(_patient_csv) if _patient_csv.exists() else pd.DataFrame()

# Organ bag attention: {case_id: {organ_idx_str: {z_s, z_e, weights}}}
ORGAN_ATTN   = json.loads(_attn_json.read_text())   if _attn_json.exists()   else {}
XORGAN_ATTN  = json.loads(_xorgan_json.read_text()) if _xorgan_json.exists() else {}

# Bilgi.xlsx — organ bag sınırları (model çıktısı yoksa)
_bilgi = DATA_ROOT / 'Bilgi.xlsx'
if _bilgi.exists():
    _bs = pd.read_excel(_bilgi, sheet_name=None)
    _t  = _bs['TRAIININGDATA'].copy()
    _t['Case Number'] = _t['Case Number'].apply(lambda x: f'T_{x}')
    _c  = _bs['COMPETITIONDATA'].copy()
    _c['Case Number']  = _c['Case Number'].apply(lambda x: f'C_{x}')
    BILGI_DF = pd.concat([_t, _c], ignore_index=True)
    BOUNDARY_DF = BILGI_DF[BILGI_DF['Type'] == 'Boundary Slice'].copy()
else:
    BILGI_DF = BOUNDARY_DF = pd.DataFrame()

print(f'FCOS tahmin  : {len(FCOS_DF):,} bbox     ({"✓" if not FCOS_DF.empty else "✗ çalıştırılmadı"})')
print(f'Hasta tahmin : {len(PATIENT_DF)} vaka    ({"✓" if not PATIENT_DF.empty else "✗ çalıştırılmadı"})')
print(f'Organ attn   : {len(ORGAN_ATTN)} vaka    ({"✓" if ORGAN_ATTN else "✗ — Bilgi.xlsx sınırları kullanılacak"})')
print(f'Cross-organ  : {len(XORGAN_ATTN)} vaka   ({"✓" if XORGAN_ATTN else "✗"})')

FCOS tahmin  : 0 bbox     (✗ çalıştırılmadı)
Hasta tahmin : 0 vaka    (✗ çalıştırılmadı)
Organ attn   : 0 vaka    (✗ — Bilgi.xlsx sınırları kullanılacak)
Cross-organ  : 0 vaka   (✗)


In [6]:
# ── PNG listesi ──────────────────────────────────────────────────────────────
_png_cache: dict = {}

def _load_png_list(case):
    if case not in _png_cache:
        pngs = []
        for split in ('train', 'val'):
            pngs.extend((FOLD_DIR / 'images' / split).glob(f'{case}_*.png'))
        _png_cache[case] = sorted(pngs, key=lambda p: int(p.stem.split('_', 1)[1]))
    return _png_cache[case]

# ── GT etiketler ─────────────────────────────────────────────────────────────
def _load_gt(case):
    rows = []
    for split in ('train', 'val'):
        for f in (FOLD_DIR / 'labels' / split).glob(f'{case}_*.txt'):
            parts = f.stem.split('_', 1)
            if len(parts) < 2:
                continue
            try:
                img_id = int(parts[1])
            except ValueError:
                continue
            for line in f.read_text().strip().splitlines():
                p = line.split()
                if len(p) < 5:
                    continue
                cls = int(p[0])
                cx, cy, w, h = map(float, p[1:5])
                rows.append({'image_id': img_id, 'class': cls,
                              'cx': cx, 'cy': cy, 'w': w, 'h': h})
    return pd.DataFrame(rows)

# ── Organ bag z-aralıkları (Bilgi.xlsx + fallback fraksiyon) ─────────────────
_DEFAULT_Z_FRACS = {
    0: (0.10, 0.90), 1: (0.30, 0.65), 2: (0.30, 0.60),
    3: (0.30, 0.85), 4: (0.40, 0.95), 5: (0.55, 0.90),
}

def _organ_z_ranges(case, n_slices):
    """Organ bag z-aralıkları: Bilgi.xlsx → fallback fraksiyon."""
    # Önce ORGAN_ATTN JSON'dan bak (model çıktısı varsa daha doğru)
    if case in ORGAN_ATTN:
        return {int(k): tuple(v[:2]) for k, v in ORGAN_ATTN[case].items()}

    result = {}
    case_boundary = BOUNDARY_DF[BOUNDARY_DF['Case Number'] == case] if not BOUNDARY_DF.empty else pd.DataFrame()

    for org_idx, org_name in enumerate(ANATOMICAL_CLASSES):
        z_pos = []
        if not case_boundary.empty:
            rows = case_boundary[case_boundary['Class'] == org_name]
            z_pos = list(rows['Image Id'].dropna().astype(int))

        if len(z_pos) >= 2:
            result[org_idx] = (min(z_pos), max(z_pos))
        else:
            fs, fe = _DEFAULT_Z_FRACS[org_idx]
            result[org_idx] = (int(fs * n_slices), min(int(fe * n_slices), n_slices - 1))

    return result

# ── Hasta tahmini ────────────────────────────────────────────────────────────
def _patient_info(case):
    """Döndürür: (probs[6], triage_label, triage_score, uncertainty) veya None."""
    if PATIENT_DF.empty:
        return None
    row = PATIENT_DF[PATIENT_DF['case_id'] == case]
    if row.empty:
        return None
    row = row.iloc[0]
    probs = np.array([row.get(f'prob_{c}', 0.0) for c in SUPER_CLASSES], dtype=np.float32)
    return probs, str(row.get('triage', '?')), float(row.get('triage_score', 0)), float(row.get('uncertainty', 0))

# ── Cross-organ attention matrix ─────────────────────────────────────────────
def _cross_organ_matrix(case):
    """Döndürür: (6,6) numpy array veya None."""
    if case in XORGAN_ATTN:
        return np.array(XORGAN_ATTN[case])
    return None

# ── Vaka → sınıf seti ────────────────────────────────────────────────────────
_fold_train = set(load_fold(FOLD, 'train'))
_fold_val   = set(load_fold(FOLD, 'val'))
_all_cases  = sorted(_fold_train | _fold_val)

def _case_classes(case):
    classes = set()
    for split in ('train', 'val'):
        for f in (FOLD_DIR / 'labels' / split).glob(f'{case}_*.txt'):
            for line in f.read_text().strip().splitlines():
                p = line.split()
                if p:
                    try:
                        classes.add(int(p[0]))
                    except ValueError:
                        pass
    return classes

_case_cls_map = {c: _case_classes(c) for c in _all_cases}
print(f'Vaka sınıf haritası: {len(_case_cls_map)} vaka')

Vaka sınıf haritası: 725 vaka


In [7]:
def obt_viewer(fold=0):
    train_cases = set(load_fold(fold, 'train'))
    val_cases   = set(load_fold(fold, 'val'))
    all_cases   = sorted(train_cases | val_cases)

    # ── Kontroller ────────────────────────────────────────────────────────
    split_dd = Dropdown(description='Split:', value='Tümü',
                        options=['Tümü', 'Train', 'Val'], layout={'width': '160px'})
    class_dd = Dropdown(description='Sınıf:', value='Tümü',
                        options=['Tümü'] + list(SUPER_CLASSES), layout={'width': '300px'})
    case_dd  = Dropdown(description='Vaka:', layout={'width': '180px'})
    slider   = IntSlider(description='Kesit:', min=0, max=1, value=0,
                         continuous_update=False, layout={'width': '520px'})
    gt_btn   = ToggleButton(value=True, description='GT (Mavi)',
                            button_style='info',   layout={'width': '120px'})
    pred_btn = ToggleButton(value=True, description='FCOS Tahmin',
                            button_style='danger', layout={'width': '130px'},
                            disabled=FCOS_DF.empty)
    organ_btn = ToggleButton(value=True, description='Organ Bag',
                             button_style='warning', layout={'width': '120px'})

    info_out  = Output()  # tek satır özet
    slice_out = Output()  # CT + kutu paneli
    case_out  = Output()  # hasta tahmini + cross-organ

    def filtered_cases():
        sp, cn = split_dd.value, class_dd.value
        cases = all_cases
        if sp == 'Train':
            cases = [c for c in cases if c in train_cases]
        elif sp == 'Val':
            cases = [c for c in cases if c in val_cases]
        if cn != 'Tümü':
            ci = SUPER_CLASSES.index(cn)
            cases = [c for c in cases if ci in _case_cls_map.get(c, set())]
        return cases

    def update_cases(_):
        cases = filtered_cases()
        case_dd.options = cases
        if cases:
            case_dd.value = cases[0]

    # ─────────────────────────────────────────────────────────────────────
    def draw_slice(z_idx, case, show_gt, show_pred, show_organ):
        pngs = _load_png_list(case)
        if not pngs or z_idx >= len(pngs):
            with slice_out:
                slice_out.clear_output(wait=True)
                print(f'⚠️  {case}: PNG bulunamadı.')
            return

        png_path = pngs[z_idx]
        try:
            img_id = int(png_path.stem.split('_', 1)[1])
        except (IndexError, ValueError):
            img_id = -1

        img  = np.array(Image.open(png_path).convert('RGB')).astype(np.float32) / 255.0
        H, W = img.shape[:2]
        n_slices = len(pngs)

        gt_df = _load_gt(case)
        z_ranges = _organ_z_ranges(case, n_slices)

        # Mevcut dilimde hangi organlar aktif?
        active_organs = [
            (oi, ANATOMICAL_CLASSES[oi])
            for oi, (zs, ze) in z_ranges.items()
            if int(zs) <= z_idx <= int(ze)
        ]

        # ── Üst panel: CT | GT | Tahmin ───────────────────────────────────
        n_cols   = 1 + int(show_gt) + int(show_pred and not FCOS_DF.empty)
        # Alt panel: z-önem haritası (her zaman)
        fig = plt.figure(figsize=(5 * n_cols, 9), facecolor='#111')
        gs  = gridspec.GridSpec(2, n_cols, figure=fig,
                                height_ratios=[5, 2], hspace=0.35, wspace=0.15)

        axes_top = [fig.add_subplot(gs[0, i]) for i in range(n_cols)]
        ax_z     = fig.add_subplot(gs[1, :])
        col = 0

        # --- Ham CT + organ sınır çerçevesi
        axes_top[col].imshow(img, vmin=0, vmax=1)
        if show_organ:
            for oi, oname in active_organs:
                axes_top[col].add_patch(mpatches.FancyBboxPatch(
                    (2, 2), W - 6, H - 6,
                    boxstyle='round,pad=2', linewidth=2.5,
                    edgecolor=ORGAN_COLORS[oi], facecolor='none', alpha=0.7
                ))
        org_txt = ', '.join(on for _, on in active_organs) or '—'
        axes_top[col].set_title(
            f'CT  z={z_idx+1}/{n_slices}\n{org_txt}',
            color='white', fontsize=8)
        axes_top[col].axis('off')
        col += 1

        # --- GT kutuları
        if show_gt:
            axes_top[col].imshow(img, vmin=0, vmax=1)
            sl_gt = gt_df[gt_df['image_id'] == img_id] if not gt_df.empty else pd.DataFrame()
            for _, r in sl_gt.iterrows():
                c = int(r['class'])
                x1 = (r.cx - r.w / 2) * W
                y1 = (r.cy - r.h / 2) * H
                axes_top[col].add_patch(mpatches.Rectangle(
                    (x1, y1), r.w * W, r.h * H,
                    linewidth=2, edgecolor='#3498db', facecolor='none'))
                axes_top[col].text(
                    x1, max(y1 - 3, 0),
                    SUPER_CLASSES[c] if c < len(SUPER_CLASSES) else str(c),
                    color='#3498db', fontsize=6, weight='bold',
                    bbox=dict(facecolor='black', alpha=0.4, pad=1))
            axes_top[col].set_title(
                f'GT  ({len(sl_gt)} kutu)  img_id={img_id}',
                color='white', fontsize=8)
            axes_top[col].axis('off')
            col += 1

        # --- FCOS tahmin kutuları
        if show_pred and not FCOS_DF.empty:
            axes_top[col].imshow(img, vmin=0, vmax=1)
            try:
                case_int = int(case.split('_')[1])
            except (IndexError, ValueError):
                case_int = -1
            sl_pred = FCOS_DF[
                (FCOS_DF['case'] == case_int) &
                (FCOS_DF['image_id'] == img_id)
            ]
            for _, r in sl_pred.iterrows():
                c = int(r['class'])
                color = CLASS_COLORS[c % len(CLASS_COLORS)]
                axes_top[col].add_patch(mpatches.Rectangle(
                    (r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1,
                    linewidth=2, edgecolor=color, facecolor='none'))
                axes_top[col].text(
                    r.x1, max(r.y1 - 3, 0),
                    f"{SUPER_CLASSES[c][:10]} {float(r.score):.2f}",
                    color=color, fontsize=6, weight='bold',
                    bbox=dict(facecolor='black', alpha=0.4, pad=1))
            axes_top[col].set_title(
                f'OBT-FCOS  ({len(sl_pred)} kutu)',
                color='white', fontsize=8)
            axes_top[col].axis('off')

        # --- Dilim önem / organ bag haritası
        ax_z.set_facecolor('#1a1a2e')
        if show_organ:
            for oi, (zs, ze) in z_ranges.items():
                ax_z.axvspan(int(zs), int(ze), alpha=0.3,
                             color=ORGAN_COLORS[oi], label=ANATOMICAL_CLASSES[oi])

        # Organ bag attention weights
        case_attn = ORGAN_ATTN.get(case, {})
        if case_attn:
            importance = np.zeros(n_slices)
            for oi_str, attn_data in case_attn.items():
                oi = int(oi_str)
                zs, ze = int(attn_data[0]), int(attn_data[1])
                w = np.array(attn_data[2]) if len(attn_data) > 2 else np.ones(ze - zs + 1)
                zi = np.clip(np.linspace(zs, ze, len(w)), 0, n_slices - 1).astype(int)
                np.add.at(importance, zi, w)
            if importance.max() > 0:
                importance /= importance.max()
            ax_z.fill_between(range(n_slices), importance, alpha=0.6, color='white')
            ax_z.plot(range(n_slices), importance, color='white', lw=1)
        else:
            # Sadece organ sınırlarını gösteren basit çubuk
            ax_z.plot([0, n_slices - 1], [0.5, 0.5], color='#555', lw=0.5, ls='--')

        # Mevcut z çizgisi
        ax_z.axvline(z_idx, color='yellow', lw=1.5, alpha=0.9)
        ax_z.set_xlim(0, n_slices)
        ax_z.set_ylim(-0.05, 1.1)
        ax_z.set_xlabel('Dilim indeksi (z)', color='white', fontsize=8)
        ax_z.tick_params(colors='white', labelsize=7)
        for sp in ax_z.spines.values():
            sp.set_color('#444')
        ax_z.set_title('Organ Bag Sınırları' +
                       (' + Attention Ağırlıkları' if case_attn else ' (Bilgi.xlsx)'),
                       color='white', fontsize=8)

        if show_organ:
            org_patches = [
                mpatches.Patch(color=ORGAN_COLORS[i], alpha=0.7, label=ANATOMICAL_CLASSES[i])
                for i in range(6)
            ]
            ax_z.legend(handles=org_patches, loc='upper right', ncol=3,
                        fontsize=6, facecolor='#222', labelcolor='white', framealpha=0.7)

        # Lejant (sınıf renkleri)
        cls_patches = [
            mpatches.Patch(color=CLASS_COLORS[i], label=f'{i}: {SUPER_CLASSES[i]}')
            for i in range(len(SUPER_CLASSES))
        ]
        fig.legend(handles=cls_patches, loc='lower center', ncol=3,
                   fontsize=6.5, facecolor='#222', labelcolor='white',
                   framealpha=0.8, bbox_to_anchor=(0.5, -0.01))

        tag = 'Train' if case in train_cases else 'Val'
        cls_names = ', '.join(
            SUPER_CLASSES[c] for c in sorted(_case_cls_map.get(case, set()))
        )
        fig.suptitle(
            f'Vaka {case}  [{tag}]  |  {cls_names or "sınıf yok"}',
            color='white', fontsize=10, fontweight='bold', y=1.01)

        with slice_out:
            slice_out.clear_output(wait=True)
            plt.show()

    # ─────────────────────────────────────────────────────────────────────
    def draw_case_panel(case):
        pat = _patient_info(case)
        xattn = _cross_organ_matrix(case)

        n_rows = 1 + int(xattn is not None)
        fig, axes = plt.subplots(n_rows, 1, figsize=(6, 4.5 * n_rows),
                                 facecolor='#111')
        if n_rows == 1:
            axes = [axes]

        # --- Hasta olasılık bar grafik
        ax_p = axes[0]
        ax_p.set_facecolor('#1a1a2e')

        if pat is not None:
            probs, triage_label, triage_score, uncertainty = pat
            bars = ax_p.barh(SUPER_CLASSES, probs,
                             color=[CLASS_COLORS[i] for i in range(6)], alpha=0.8)
            ax_p.axvline(0.5, color='yellow', ls='--', lw=1.5, alpha=0.8,
                         label='Eşik 0.5')
            # Değerleri bar üstüne yaz
            for bar, p in zip(bars, probs):
                ax_p.text(min(p + 0.02, 0.97), bar.get_y() + bar.get_height() / 2,
                          f'{p:.2f}', va='center', ha='left',
                          color='white', fontsize=8, weight='bold')
            tc = TRIAGE_COLORS.get(triage_label, '#aaa')
            ax_p.set_title(
                f'Hasta Düzeyi Tahmin  |  '
                f'Triyaj: {triage_label} ({triage_score:.2f})  |  '
                f'Belirsizlik: {uncertainty:.2f}',
                color=tc, fontsize=9, fontweight='bold')
        else:
            ax_p.text(0.5, 0.5, 'Hasta tahmini yok\n(inference çalıştırılmadı)',
                      ha='center', va='center', color='#aaa', fontsize=10,
                      transform=ax_p.transAxes)
            ax_p.set_title('Hasta Düzeyi Tahmin', color='white', fontsize=9)

        ax_p.set_xlim(0, 1.1)
        ax_p.set_xlabel('Olasılık', color='white', fontsize=8)
        ax_p.tick_params(colors='white', labelsize=8)
        for sp in ax_p.spines.values():
            sp.set_color('#444')

        # --- Cross-organ attention matrisi
        if xattn is not None:
            ax_x = axes[1]
            ax_x.set_facecolor('#1a1a2e')
            im = ax_x.imshow(xattn, cmap='Blues', vmin=0, vmax=xattn.max())
            short = [c.split()[0] for c in ANATOMICAL_CLASSES]
            ax_x.set_xticks(range(6))
            ax_x.set_yticks(range(6))
            ax_x.set_xticklabels(short, fontsize=7, color='white')
            ax_x.set_yticklabels(short, fontsize=7, color='white')
            for i in range(6):
                for j in range(6):
                    ax_x.text(j, i, f'{xattn[i,j]:.2f}', ha='center', va='center',
                              fontsize=7,
                              color='white' if xattn[i,j] > xattn.max() * 0.6 else '#ccc')
            plt.colorbar(im, ax=ax_x, shrink=0.8)
            ax_x.set_title('Cross-Organ Attention Matrisi', color='white', fontsize=9)
            for sp in ax_x.spines.values():
                sp.set_color('#444')

        plt.tight_layout()
        with case_out:
            case_out.clear_output(wait=True)
            plt.show()

    # ─────────────────────────────────────────────────────────────────────
    def on_case_change(_):
        case = case_dd.value
        if case is None:
            return
        pngs = _load_png_list(case)
        n = len(pngs)
        slider.max   = max(0, n - 1)
        slider.value = n // 2

        with info_out:
            info_out.clear_output()
            tag   = 'Train' if case in train_cases else 'Val'
            gt_df = _load_gt(case)
            n_ann = gt_df['image_id'].nunique() if not gt_df.empty else 0
            n_box = len(gt_df)
            pat   = _patient_info(case)
            tr_str = f"triyaj={pat[1]} ({pat[2]:.2f})  bel.={pat[3]:.2f}" if pat else 'tahmin yok'
            print(
                f'Vaka {case}  [{tag}]  |  {n} kesit  |  '
                f'{n_ann} ann. dilim  |  {n_box} GT kutu  |  {tr_str}  |  '
                f'Sınıflar: {chr(44).join(SUPER_CLASSES[c] for c in sorted(_case_cls_map.get(case, set()))) or "—"}'
            )

        draw_slice(slider.value, case, gt_btn.value, pred_btn.value, organ_btn.value)
        draw_case_panel(case)

    def on_slice_change(_):
        case = case_dd.value
        if case is not None:
            draw_slice(slider.value, case, gt_btn.value, pred_btn.value, organ_btn.value)

    def on_toggle(_):
        on_slice_change(None)

    for w in [split_dd, class_dd]:
        w.observe(update_cases, names='value')
    case_dd.observe(on_case_change, names='value')
    slider.observe(on_slice_change, names='value')
    for w in [gt_btn, pred_btn, organ_btn]:
        w.observe(on_toggle, names='value')

    display(VBox([
        HBox([split_dd, class_dd]),
        HBox([case_dd, gt_btn, pred_btn, organ_btn]),
        info_out,
        slider,
        HBox([slice_out, case_out]),
    ]))
    threading.Timer(0.3, update_cases, args=[None]).start()

In [8]:
obt_viewer(fold=FOLD)